Process the Bronze tables data

In [0]:
#Initialize
import uuid
from pyspark.sql import functions as F
from datetime import datetime

catalog_name = "workspace"
database_name = "amazon_fullfilment_bronze"

bronze_volume_path ="/Volumes/workspace/amazon_fullfilment_bronze/landing_zone/"
silver_volume_path ="/Volumes/workspace/amazon_fullfilment_silver/processing_zone/"
source_volume_path = "/Volumes/workspace/default/amazon_fullfilment/source/"

# Generate a unique ID for this specific notebook run
#current_run_uuid = str(uuid.uuid4())
# 93f7758a-f5a2-4b6f-bf40-da5b67878c02
current_run_uuid = "2"

def add_bronze_metadata(df):
    """
    Standardizes the metadata for all Bronze tables.
    """
    return df.withColumn("_ingest_timestamp", F.current_timestamp()) \
             .withColumn("_source_file", F.col("_metadata.file_path")) \
             .withColumn("_batch_id", F.lit(current_run_uuid))

def get_current_shift():
    # Get the current hour (0-23)
    current_hour = datetime.now().hour
    
    if 0 <= current_hour < 8:
        return 'Overnight'
    elif 8 <= current_hour < 16:
        return 'Morning'
    else:
        return 'Afternoon'

current_shift = get_current_shift()



In [0]:
# Insert into Bronze layer function
 
def ingest_to_bronze(table_name, schema, source_path, checkpoint_path):
    """
    Standardizes the Auto Loader ingestion for any table.
    """
    raw_stream = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .schema(schema)
        .load(source_path))
    
    # We select * and _metadata to ensure the global function works
    return (add_bronze_metadata(raw_stream.select("*", "_metadata"))
        .writeStream
        .option("checkpointLocation", f"{checkpoint_path}/_checkpoint")
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(f"{catalog_name}.{database_name}.{table_name}"))

In [0]:
# Generate Orders data
# Generates random number of orders depending of the time of the day

from pyspark.sql import Window, functions as F
from pyspark.sql.types import StringType
import uuid
import pytz
from datetime import datetime
import random

tz = pytz.timezone('America/New_York')
def get_rand_num_orders():
    # Get the current hour (0-23)
    current_time_est = datetime.now(tz)
    current_hour = current_time_est.hour

    if 0 <= current_hour < 6:
        return 0
    elif 6 <= current_hour < 9:
        return random.randint(0,10)
    elif 9 <= current_hour < 15:
        return random.randint(0,5)
    else:
        return random.randint(0,50)
    
# 1. Get the number of orders (Your existing Python logic is fine here)
num_orders = get_rand_num_orders()
display(num_orders)

def generate_orders_spark(num_orders):
    if num_orders == 0:
        # Returns empty DF with correct schema
        return spark.createDataFrame([], "order_id string, status string, updated_timestamp timestamp, orderdate date, customer_id string")

    # 2. Create the base orders (Parallelized)
    # We generate the metadata for the orders first
    base_orders = spark.range(0, num_orders).select(
        F.expr("uuid()").alias("order_id"),
        F.expr("date_sub(current_date(), cast(rand() * 3 as int))").alias("orderdate"),
        F.lit("Initial").alias("status"),
        F.current_timestamp().alias("updated_timestamp"),
        F.row_number().over(Window.orderBy(F.monotonically_increasing_id())).alias("join_key")
    )

    # 3. Get a random sample of customers from the millions available
    # We only need as many customers as we have orders
    # We use a window to give them a matching join_key
    customers = (spark.table("workspace.amazon_fullfilment_silver.customer_silver")
                 .select("customer_id")
                 .sample(withReplacement=False, fraction=0.1) # Sample 10% to speed up shuffle
                 .withColumn("cust_rand", F.rand())
                 .withColumn("join_key", F.row_number().over(Window.orderBy("cust_rand")))
                 .filter(F.col("join_key") <= num_orders))

    # 4. Join on the incrementing key
    # This is a highly efficient 1-to-1 join
    final_orders = (base_orders.join(customers, "join_key", "inner")
                    .drop("join_key", "cust_rand"))

    return final_orders

# Generate and view
orders_df_raw = generate_orders_spark(num_orders)
orders_df = orders_df_raw.select("order_id", "customer_id" , "orderdate", "status", "updated_timestamp")
#display(orders_df)


#save to the source volume
(orders_df.write
.format("csv")
.option("header",True)
.option("delimiter",",")
.mode("append")
.save(f"{source_volume_path}/orders"))

In [0]:
# Insert into Bronze layer order table
 
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType, IntegerType

# Define exactly what the data should look like
orders_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("orderdate", DateType(), True),
    StructField("status", StringType(), False),
    StructField("updated_timestamp", TimestampType(), False)
])

# 1. Define your paths
bronze_orders_path = f"{bronze_volume_path}orders"

ingest_to_bronze("orders", orders_schema, f"{source_volume_path}orders", bronze_orders_path)

In [0]:
# process new records into the Silver layer orders table
from pyspark.sql import functions as F

def process_scd2_orders(microBatchDF, batchId):
    # 1. Get a list of order_ids that already exist in Silver
    # This prevents duplicating brand new orders
    target_table = "workspace.amazon_fullfilment_silver.orders_silver"
    
    # 2. Split the micro-batch logic
    # Part A: The 'Insert' half (Always NULL merge_key)
    inserts_df = microBatchDF.withColumn("merge_key", F.lit(None))
    
    # Part B: The 'Update' half (Only for IDs that already exist in Silver)
    # This prevents the 'double insert' on the first run
    existing_ids = microBatchDF.sparkSession.table(target_table) \
        .filter("is_current = true") \
        .select("order_id")
    
    updates_df = microBatchDF.join(existing_ids, "order_id", "inner") \
        .withColumn("merge_key", F.col("order_id"))
    
    # 3. Combine and Merge
    combined_df = inserts_df.unionByName(updates_df, allowMissingColumns=True)
    combined_df.createOrReplaceTempView("source_view")

    microBatchDF.sparkSession.sql(f"""
        MERGE INTO {target_table} AS target
        USING source_view AS source
        ON target.order_id = source.merge_key AND target.is_current = true
        
        -- If we find the ID and status changed, expire the old one
        WHEN MATCHED AND target.status <> source.status THEN
          UPDATE SET target.is_current = false, target.end_date = source.updated_timestamp
          
        -- If it's the 'NULL' half of the union, it will always be NOT MATCHED -> Insert
        WHEN NOT MATCHED THEN
          INSERT (order_sk, order_id, customer_id, orderdate, status, updated_timestamp, is_current, start_date, end_date)
          VALUES (md5(concat(source.order_id, cast(source.updated_timestamp as STRING))), 
                  source.order_id, source.customer_id, source.orderdate, source.status, 
                  source.updated_timestamp, true, source.updated_timestamp, NULL)
    """)

    # 3. Start the Stream
query = (spark.readStream
    .table("workspace.amazon_fullfilment_bronze.orders")
    .filter("status IS NOT NULL") # Basic data quality
    .writeStream
    .foreachBatch(process_scd2_orders)
    .option("checkpointLocation", f"{bronze_volume_path}/checkpoints/silver_orders_scd2")
    .trigger(availableNow=True) # Or .trigger(processingTime='1 minute') for 24/7
    .start())

In [0]:
# Generate Orders_Item data
import pytz
from datetime import datetime

import uuid
import random
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, IntegerType
from pyspark.sql import functions as F

orders_df = spark.table("workspace.amazon_fullfilment_silver.orders_silver").select("order_id").filter("status='Initial' AND is_current=true").alias("orders")
products_df = spark.table("workspace.amazon_fullfilment_silver.products_silver").select("product_id").filter("is_current=true").alias("products")


tz = pytz.timezone('America/New_York')

num_orders =orders_df.count()

def get_rand_num_order_items():
    if num_orders == 0:
        return 0
    else:
        return random.randint(num_orders,num_orders*2)


num_order_items = get_rand_num_order_items()
def generate_order_items_data(num_order_items=0):
# 2. Create a base of random numbers and join
    # This creates a "random sample" across your available orders and products
    return orders_df.crossJoin(products_df) \
        .orderBy(F.rand()) \
        .limit(num_order_items) \
        .withColumn("order_item_id", F.expr("uuid()")) \
        .withColumn("quantity", (F.rand() * 10 + 1).cast("int")) \
        .select(
            F.col("orders.order_id"), 
            F.col("order_item_id"), 
            F.col("products.product_id"), 
            F.col("quantity"))


# Generate and view
order_items_df = generate_order_items_data(num_order_items)

# save to the source volume

(order_items_df.write
.format("csv")
.option("header",True)
.option("delimiter",",")
.mode("append")
.save(f"{source_volume_path}/order_items"))

In [0]:
# Insert into Bronze layer order_items table
 
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType, IntegerType

# Define exactly what the data should look like
order_item_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("order_item_id", StringType(), True),
    StructField("product_id", StringType(), False),
    StructField("quantity", IntegerType(), True)
])

# 1. Define your paths
bronze_order_items_path = f"{bronze_volume_path}order_items"

ingest_to_bronze("order_items", order_item_schema, f"{source_volume_path}order_items", bronze_order_items_path)

In [0]:
# process new records into the Silver layer order_items table
from pyspark.sql import functions as F

def process_scd1_order_items(microBatchDF, batchId):
    # 1. Get a list of order_ids that already exist in Silver
    # This prevents duplicating brand new orders
    target_table = "workspace.amazon_fullfilment_silver.order_items_silver"
    
    # 2. Split the micro-batch logic
    # Part A: The 'Insert' half (Always NULL merge_key)
    inserts_df = microBatchDF.withColumn("merge_key", F.lit(None))
    
    # Part B: The 'Update' half (Only for IDs that already exist in Silver)
    # This prevents the 'double insert' on the first run
    existing_ids = microBatchDF.sparkSession.table(target_table).select("order_item_id")
       # .filter("is_current = true") \
    
    
    updates_df = microBatchDF.join(existing_ids, "order_item_id", "inner") \
        .withColumn("merge_key", F.col("order_item_id"))
    
    # 3. Combine and Merge
    combined_df = inserts_df.unionByName(updates_df, allowMissingColumns=True)
    #display(combined_df)
    combined_df.createOrReplaceTempView("source_order_items_view")

    microBatchDF.sparkSession.sql(f"""
        MERGE INTO {target_table} AS target
        USING source_order_items_view AS source
        ON target.order_item_id = source.merge_key
        
        -- If we find the ID and status changed, expire the old one
        WHEN MATCHED AND source.product_id <> target.product_id AND source.quantity <> target.quantity THEN
            UPDATE SET 
            target.product_id = source.product_id,
            target.quantity = source.quantity,
            target.updated_timestamp = current_timestamp()
          
        -- If it's the 'NULL' half of the union, it will always be NOT MATCHED -> Insert
        WHEN NOT MATCHED THEN
          INSERT (
            order_item_sk, 
            order_item_id, 
            order_id, 
            product_id, 
            quantity, 
            unit_price, 
            updated_timestamp
          )
          VALUES (
            md5(source.order_item_id), 
            source.order_item_id, 
            source.order_id, 
            source.product_id, 
            source.quantity, 
            0.00, 
            current_timestamp()
          )
    """)

    # 3. Start the Stream
query = (spark.readStream
    .table("workspace.amazon_fullfilment_bronze.order_items")
    .filter("order_item_id IS NOT NULL") # Basic data quality
    .writeStream
    .foreachBatch(process_scd1_order_items)
    .option("checkpointLocation", f"{silver_volume_path}/checkpoints/silver_order_items_scd1")
    .trigger(availableNow=True) # Or .trigger(processingTime='1 minute') for 24/7
    .start())

In [0]:
%sql
select * from workspace.amazon_fullfilment_silver.orders_silver 
join workspace.amazon_fullfilment_silver.order_items_silver on orders_silver.order_id=order_items_silver.order_id
join workspace.amazon_fullfilment_silver.products_silver on products_silver.product_id=order_items_silver.product_id and products_silver.is_current=true
where orders_silver.status='Initial' AND orders_silver.is_current=true 

In [0]:
# fill shelves and inventories based on the new or existing orders and order items. Then update order and robots records
from pyspark.sql import Window
import pyspark.sql.functions as F
import uuid


def generate_shelves_inventory_data():
    # 1. Get detailed order items with weights
    # We need weights at the item level to calculate cumulative sum accurately
    order_item_details = (spark.table("workspace.amazon_fullfilment_silver.orders_silver")
        .filter("status='Initial' AND is_current=true")
        .join(spark.table("workspace.amazon_fullfilment_silver.order_items_silver"), "order_id")
        .join(spark.table("workspace.amazon_fullfilment_silver.products_silver").filter("is_current=true"), "product_id")
        .select("order_id", "product_id", "quantity", "customer_id", "orderdate", (F.col("quantity") * F.col("weight_kg")).alias("item_total_weight"))
    )

    # 2. Calculate Cumulative Weight to assign "Shelf Groups"
    # We group items until they hit the 30kg threshold
    running_weight_window = Window.orderBy("order_id", "product_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)
    
    # assign_idx determines which "30kg bucket" the item falls into
    order_buckets = order_item_details.withColumn("cumulative_weight", F.sum("item_total_weight").over(running_weight_window)) \
        .withColumn("shelf_rank", F.floor(F.col("cumulative_weight") / 30.0).cast("int"))

    # 3. Get Available Shelves and Robots (Unique pairs)
    # Join Robots to their Active Shelves first
    shelf_pool = (spark.table("workspace.amazon_fullfilment_silver.shelves")
                  .filter("is_active = true AND status = 'Idle'")
                  .alias("shlf")
                  .join(spark.table("workspace.amazon_fullfilment_silver.robots").alias("rb"), "robot_id")
                  .filter("rb.is_active = true AND  rb.status = 'Idle' AND rb.current_battery_pct > 10")
                  # Ensure we only get the latest event per robot
                  .withColumn("rn", F.row_number().over(Window.partitionBy("robot_id").orderBy(F.col("rb.Updated_Timestamp").desc())))
                  .filter("rn = 1")
                  .drop("rn")
                  .withColumn("available_shelf_rank", F.row_number().over(Window.orderBy(F.col("current_battery_pct").desc())))
                  # Adjust rank to start at 0 to match our F.floor index
                  .withColumn("available_shelf_rank", F.col("available_shelf_rank") - 1)
    )

    # 4. Final Assignment Join
    # Match the "Shelf Group" (0, 1, 2...) with the "Available Shelf Rank"
    assignments = order_buckets.join(shelf_pool, F.col("shelf_rank") == F.col("available_shelf_rank"), "inner")

    # --- CREATE OUTPUT DATAFRAMES ---

    # A. Shelves (Now unique per shelf_id)
    shelves_df = assignments.select(
        "shelf_id", "robot_id", "shlf.max_weight_capacity",
        F.lit("Picking").alias("status"),
        F.current_timestamp().alias("updated_timestamp")
    ).distinct()

    # B. Inventory (Multiple items can now share the same shelf_id)
    inventory_df = assignments.select(
        "shelf_id", "product_id", "quantity", "order_id"
    )

    # C. Robot Updates
    robot_updates_df = assignments.select(
        "robot_id",
        F.lit("Picking").alias("status"),
        F.lit(500.0).alias("max_weight_capacity"),
        "current_battery_pct",
        F.current_timestamp().alias("Updated_Timestamp")
    ).distinct()

    # D. Order Status Updates
    current_run_uuid = str(uuid.uuid4())

    order_status_updates_df = assignments.select(
        "order_id",
        "customer_id",
        "orderdate",
        F.lit("Picking").alias("status"),
        F.current_timestamp().alias("updated_timestamp")
    ).distinct()

    return shelves_df, inventory_df, robot_updates_df, order_status_updates_df

# Execute and Save
shelves_df, inventory_df, robot_updates_df, order_status_updates_df = generate_shelves_inventory_data()
display(shelves_df)
display(inventory_df)
display(robot_updates_df)
display(order_status_updates_df)

# save to the source volume

(shelves_df.write
.format("csv")
.option("header",True)
.option("delimiter",",")
.mode("append")
.save(f"{source_volume_path}/shelves_events"))
(inventory_df.write
.format("csv")
.option("header",True)
.option("delimiter",",")
.mode("append")
.save(f"{source_volume_path}/inventory"))
(robot_updates_df.write
.format("csv")
.option("header",True)
.option("delimiter",",")
.mode("append")
.save(f"{source_volume_path}/robot_events"))
(order_status_updates_df.write
.format("csv")
.option("header",True)
.option("delimiter",",")
.mode("append")
.save(f"{source_volume_path}/orders"))

In [0]:
# Insert into Bronze layer Shelves table
 
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType, IntegerType

# Define exactly what the data should look like
shelve_schema = StructType([
    StructField("shelf_id", StringType(), True),
    StructField("robot_id", StringType(), True),
    StructField("max_weight_capacity", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("updated_timestamp", TimestampType(), True)
])

# 1. Define your paths
bronze_shelves_path = f"{bronze_volume_path}shelves_events"

ingest_to_bronze("shelves_events", shelve_schema, f"{source_volume_path}shelves_events", bronze_shelves_path)

In [0]:
# process new records into the Silver layer shelves table
from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp, current_date

def process_scd2_shelves(microBatchDF, batchId):
    # 1. Get a list of order_ids that already exist in Silver
    # This prevents duplicating brand new orders
    target_table = "workspace.amazon_fullfilment_silver.shelves"
    
    batch_df = microBatchDF.orderBy(F.col("updated_timestamp").desc()) \
                           .dropDuplicates(["shelf_id"])
    # 2. Split the micro-batch logic
    # Part A: The 'Insert' half (Always NULL merge_key)
    inserts_df = batch_df.withColumn("merge_key", F.lit(None))
    
    # Part B: The 'Update' half (Only for IDs that already exist in Silver)
    # This prevents the 'double insert' on the first run
    existing_ids = microBatchDF.sparkSession.table(target_table) \
        .filter("is_active = true") \
        .select("shelf_id")
    
    updates_df = batch_df.join(existing_ids, "shelf_id", "inner") \
        .withColumn("merge_key", F.col("shelf_id"))
    
    # 3. Combine and Merge
    combined_df = inserts_df.unionByName(updates_df, allowMissingColumns=True)
    #combined_df.createOrReplaceTempView("source_shelves_view")
    view_name = f"source_shelves_{batchId}"
    combined_df.createOrReplaceTempView(view_name)
    try:
        microBatchDF.sparkSession.sql(f"""
            MERGE INTO {target_table} AS target
            USING {view_name} AS source
            ON target.shelf_id = source.merge_key AND target.is_active = true
            
            -- If we find the ID and status changed, expire the old one
            WHEN MATCHED AND (target.status <> source.status OR NOT(target.robot_id <=> source.robot_id)) THEN
            UPDATE SET target.is_active = false, target.end_date = source.updated_timestamp, target.updated_timestamp = source.updated_timestamp
            
            -- If it's the 'NULL' half of the union, it will always be NOT MATCHED -> Insert
            WHEN NOT MATCHED THEN
            INSERT (shelf_sk, shelf_id, robot_id, max_weight_capacity, status, is_active, start_date, end_date, updated_timestamp)
            VALUES (md5(concat(source.shelf_id, cast(source.updated_timestamp as STRING))), 
                    source.shelf_id, source.robot_id, source.max_weight_capacity, source.status,  
                    true, source.updated_timestamp, NULL, source.updated_timestamp)
        """)
    except Exception as e:
        print(f"Error in batch {batchId}: {e}")
        raise e

    microBatchDF.sparkSession.catalog.dropTempView(view_name)
    # 3. Start the Stream
query = (spark.readStream
    .table("workspace.amazon_fullfilment_bronze.shelves_events")
    .filter("status IS NOT NULL") # Basic data quality
    .writeStream
    .foreachBatch(process_scd2_shelves)
    .option("checkpointLocation", f"{silver_volume_path}/checkpoints/silver_shelves_scd2")
    .trigger(availableNow=True) # Or .trigger(processingTime='1 minute') for 24/7
    .start())

In [0]:
%sql
select count(*), status from workspace.amazon_fullfilment_silver.shelves group by status

In [0]:
# Insert into Bronze layer Inventory table
 
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType, IntegerType

# Define exactly what the data should look like
inventory_schema = StructType([
    StructField("shelf_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("order_id", StringType(), True)
])

# 1. Define your paths
bronze_inventory_path = f"{bronze_volume_path}inventory"

ingest_to_bronze("inventory", inventory_schema, f"{source_volume_path}inventory", bronze_inventory_path)

In [0]:
# process new records into the Silver layer inventory table
from pyspark.sql import functions as F

def process_scd1_inventory(microBatchDF, batchId):
    target_table = "workspace.amazon_fullfilment_silver.inventory_silver"
    
    # 1. Deduplicate based on the COMPOSITE natural key
    # This ensures we only have one row per shelf/product combo per batch
    source_df = microBatchDF.dropDuplicates(["shelf_id", "product_id"])
    source_df.createOrReplaceTempView("source_inventory_view")

    # 2. Updated Merge Logic
    microBatchDF.sparkSession.sql(f"""
        MERGE INTO {target_table} AS target
        USING source_inventory_view AS source
        ON target.shelf_id = source.shelf_id 
           AND target.product_id = source.product_id
        
        -- We only update the 'Quantity' or 'Order_ID' if the composite key matches
        WHEN MATCHED AND (
            source.quantity <> target.quantity OR 
            source.order_id <> target.order_id
        ) THEN
            UPDATE SET 
                target.quantity = source.quantity,
                target.order_id = source.order_id,
                target.updated_timestamp = current_timestamp()
          
        WHEN NOT MATCHED THEN
          INSERT (
            inventory_sk, 
            shelf_id, 
            product_id, 
            quantity, 
            order_id, 
            updated_timestamp
          )
          VALUES (
            -- IMPORTANT: The SK must now be a hash of BOTH columns
            md5(concat(source.shelf_id, source.product_id)), 
            source.shelf_id, 
            source.product_id, 
            source.quantity, 
            source.order_id, 
            current_timestamp()
          )
    """)
    # 3. Start the Stream
query = (spark.readStream
    .table("workspace.amazon_fullfilment_bronze.inventory")
    .filter("shelf_id IS NOT NULL") # Basic data quality
    .writeStream
    .foreachBatch(process_scd1_inventory)
    .option("checkpointLocation", f"{silver_volume_path}/checkpoints/silver_inventory_scd1")
    .trigger(availableNow=True) # Or .trigger(processingTime='1 minute') for 24/7
    .start())

In [0]:
%sql
select * from workspace.amazon_fullfilment_silver.inventory_silver

In [0]:
# Insert into Bronze layer robot_events table
 
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType, IntegerType

# Define exactly what the data should look like
robot_events_schema = StructType([
    StructField("robot_id", StringType(), True),
    StructField("status", StringType(), True),
    StructField("max_weight_capacity", DoubleType(), True),
    StructField("battery_level", IntegerType(), True),
    StructField("event_timestamp", TimestampType(), True)
])

# 1. Define your paths
bronze_robot_events_path = f"{bronze_volume_path}robot_events"

ingest_to_bronze("robot_events", robot_events_schema, f"{source_volume_path}robot_events", bronze_robot_events_path)

In [0]:
# process new records into the Silver layer robots table
from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp, current_date
from pyspark.sql import Window


def process_scd2_robots(microBatchDF, batchId):
    # 1. Get a list of order_ids that already exist in Silver
    # This prevents duplicating brand new orders
    target_table = "workspace.amazon_fullfilment_silver.robots"
    
    window_spec = Window.partitionBy("robot_id").orderBy(F.col("event_timestamp").desc())
    
    # This keeps only the most recent status for each robot in the current micro-batch
    deduped_df = microBatchDF \
        .withColumn("rn", F.row_number().over(window_spec)) \
        .filter("rn = 1") \
        .drop("rn")
    
    # 2. Split the micro-batch logic using the DEDUPED data
    # Part A: The 'Insert' half
    inserts_df = deduped_df.withColumn("merge_key", F.lit(None))
    
    # Part B: The 'Update' half
    existing_ids = deduped_df.sparkSession.table(target_table) \
        .filter("is_active = true") \
        .select("robot_id")
    
    updates_df = deduped_df.join(existing_ids, "robot_id", "inner") \
        .withColumn("merge_key", F.col("robot_id"))
    
    # 3. Combine and Merge
    combined_df = inserts_df.unionByName(updates_df, allowMissingColumns=True)
    
    # Register the view from the combined and cleaned dataframe
    combined_df.createOrReplaceTempView("source_robot_view")

  
    microBatchDF.sparkSession.sql(f"""
        MERGE INTO {target_table} AS target
        USING source_robot_view AS source
        ON target.robot_id = source.merge_key AND target.is_active = true
        
        -- If we find the ID and status changed, expire the old one
        WHEN MATCHED AND target.status <> source.status THEN
          UPDATE SET target.is_active = false, target.end_date = source.event_timestamp, target.updated_timestamp = source.event_timestamp
          
        -- If it's the 'NULL' half of the union, it will always be NOT MATCHED -> Insert
        WHEN NOT MATCHED THEN
          INSERT (robot_sk, robot_id, robot_type, max_weight_capacity, max_speed_mps, status, current_battery_pct, last_maintenance_date,  is_active, start_date, end_date, updated_timestamp)
          VALUES (md5(concat(source.robot_id, cast(source.event_timestamp as STRING))), 
                  source.robot_id, 'Hercules', source.max_weight_capacity, 10.0, source.status, source.battery_level, current_date(), 
                  true, source.event_timestamp, NULL, source.event_timestamp)
    """)

    # 3. Start the Stream
query = (spark.readStream
    .table("workspace.amazon_fullfilment_bronze.robot_events")
    .filter("status IS NOT NULL") # Basic data quality
    .writeStream
    .foreachBatch(process_scd2_robots)
    .option("checkpointLocation", f"{silver_volume_path}/checkpoints/silver_robots_scd2")
    .trigger(availableNow=True) # Or .trigger(processingTime='1 minute') for 24/7
    .start())

In [0]:
# Insert into Bronze layer order table
 
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType, IntegerType

# Define exactly what the data should look like
orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("orderdate", DateType(), True),
    StructField("status", StringType(), True),
    StructField("updated_timestamp", TimestampType(), True)
])

# 1. Define your paths
bronze_orders_path = f"{bronze_volume_path}orders"

ingest_to_bronze("orders", orders_schema, f"{source_volume_path}orders", bronze_orders_path)

In [0]:
%sql
select status, count(*) from workspace.amazon_fullfilment_silver.orders_silver group by status
  

In [0]:
%sql
select status, count(*) from workspace.amazon_fullfilment_bronze.orders group by status

In [0]:
%sql
select * from workspace.amazon_fullfilment_bronze.orders where status ='Picking'

In [0]:
# process new records into the Silver layer orders table
from pyspark.sql import functions as F

def process_scd2_orders(microBatchDF, batchId):
    # 1. Get a list of order_ids that already exist in Silver
    # This prevents duplicating brand new orders
    target_table = "workspace.amazon_fullfilment_silver.orders_silver"
    
    # 2. Split the micro-batch logic
    # Part A: The 'Insert' half (Always NULL merge_key)
    inserts_df = microBatchDF.withColumn("merge_key", F.lit(None))
    
    # Part B: The 'Update' half (Only for IDs that already exist in Silver)
    # This prevents the 'double insert' on the first run
    existing_ids = microBatchDF.sparkSession.table(target_table) \
        .filter("is_current = true") \
        .select("order_id")
    
    updates_df = microBatchDF.join(existing_ids, "order_id", "inner") \
        .withColumn("merge_key", F.col("order_id"))
    
    # 3. Combine and Merge
    combined_df = inserts_df.unionByName(updates_df, allowMissingColumns=True)
    combined_df.createOrReplaceTempView("source_view")

    microBatchDF.sparkSession.sql(f"""
        MERGE INTO {target_table} AS target
        USING source_view AS source
        ON target.order_id = source.merge_key AND target.is_current = true
        
        -- If we find the ID and status changed, expire the old one
        WHEN MATCHED AND target.status <> source.status THEN
          UPDATE SET target.is_current = false, target.end_date = source.updated_timestamp
          
        -- If it's the 'NULL' half of the union, it will always be NOT MATCHED -> Insert
        WHEN NOT MATCHED THEN
          INSERT (order_sk, order_id, customer_id, orderdate, status, updated_timestamp, is_current, start_date, end_date)
          VALUES (md5(concat(source.order_id, cast(source.updated_timestamp as STRING))), 
                  source.order_id, source.customer_id, source.orderdate, source.status, 
                  source.updated_timestamp, true, source.updated_timestamp, NULL)
    """)

    # 3. Start the Stream
query = (spark.readStream
    .table("workspace.amazon_fullfilment_bronze.orders")
    .filter("status IS NOT NULL") # Basic data quality
    .writeStream
    .foreachBatch(process_scd2_orders)
    .option("checkpointLocation", f"{bronze_volume_path}/checkpoints/silver_orders_scd2")
    .trigger(availableNow=True) # Or .trigger(processingTime='1 minute') for 24/7
    .start())

In [0]:
# Generating Bin records

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DecimalType

# --- 1. RESOURCE CALCULATION & CONSTRAINTS ---
available_bins_df = (spark.table("workspace.amazon_fullfilment_silver.bins")
                     .filter("status = 'idle' AND is_active = true"))

employees_count = (spark.table("workspace.amazon_fullfilment_silver.employee")
                   .filter(f"is_active = true AND job_role = 'Stower' AND shift_id = '{current_shift}'")
                   .count())

# Calculate capacity based on employees (1 employee : 5 bins)
employee_bin_capacity = employees_count * 5
# Total bins available in the warehouse (capped at 100 as per your requirement)
warehouse_bin_limit = min(available_bins_df.count(), 100)

# The final bottleneck constraint
max_bins_to_process = min(employee_bin_capacity, warehouse_bin_limit)

print(f"Capacity: {employees_count} employees can handle {employee_bin_capacity} bins. "
      f"Warehouse has {warehouse_bin_limit} idle bins. "
      f"Processing Limit: {max_bins_to_process} bins.")

# --- 2. DATA PREPARATION (The Join & Explode) ---
orders_in_picking = (spark.table("workspace.amazon_fullfilment_silver.orders_silver")
    .filter("is_current = true AND status = 'Picking'")
    .select("order_id"))

items_to_stow = (spark.table("workspace.amazon_fullfilment_silver.order_items_silver").alias("order_items")
    .join(orders_in_picking, "order_id")
    .join(spark.table("workspace.amazon_fullfilment_silver.products_silver").filter("is_current=true"), "product_id")
    .withColumn("quantity_array", F.expr("explode(sequence(1, quantity))"))
    .withColumn("unique_item_id", F.monotonically_increasing_id())
    .select("unique_item_id", "order_items.order_id", "weight_kg"))

# --- 3. VIRTUAL BINNING ---
strict_window = Window.orderBy("unique_item_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)

bin_calculation = (items_to_stow
    .withColumn("running_total", F.sum("weight_kg").over(strict_window))
    .withColumn("virtual_bin_id", F.floor((F.col("running_total") - F.lit(0.001)) / 60.0)))

# --- 4. APPLY THE WORK LIMIT ---
# We group and then limit the number of virtual bins based on our constraint
virtual_bins_final = (bin_calculation.groupBy("virtual_bin_id")
    .agg(
        F.sum("weight_kg").alias("bin_total_weight"),
        # Change: Convert array of IDs to a comma-separated string for CSV compatibility
        F.concat_ws(", ", F.collect_set("order_id")).alias("formatted_order_ids")
    )
    .orderBy("virtual_bin_id")
    .limit(max_bins_to_process)) # APPLY EMPLOYEE/WAREHOUSE CONSTRAINT HERE

# --- 5. PHYSICAL MAPPING & OUTPUT ---
available_bins_ranked = available_bins_df.withColumn("bin_rank", F.row_number().over(Window.orderBy("bin_id")))
ranked_virtual_bins = virtual_bins_final.withColumn("v_rank", F.row_number().over(Window.orderBy("virtual_bin_id")))

bin_assignments_df = (ranked_virtual_bins.join(available_bins_ranked, ranked_virtual_bins.v_rank == available_bins_ranked.bin_rank)
    .withColumn("status", 
        F.when((F.col("bin_total_weight") > 50) , "Shipment")
        .otherwise("Picking"))
    .select(
        "bin_id",
        F.col("formatted_order_ids").alias("order_id"),
        F.lit(0).alias("item_count"),
         "status",  
        F.col("bin_total_weight").cast("double").alias("current_weight"), 
        F.lit("N/A").alias("employee_id"),
        F.current_timestamp().alias("updated_timestamp"),
        F.lit(current_run_uuid).alias("_batch_id"),
        F.current_timestamp().alias("_ingest_timestamp")
    ))

#display(bin_assignments_df)

(bin_assignments_df
 .drop("_metadata")
 .write.format("csv")
 .option("header",True)
 .option("delimiter",",")
 .mode("append")
 .save(f"{source_volume_path}/bin_events"))

In [0]:
display(bin_assignments_df)

In [0]:
%sql
select * from workspace.amazon_fullfilment_silver.order_items_silver where order_id in ('e8905e2f-aab5-4bbd-9b5b-b9a4089330b1', '43eb70ec-ff11-4cf8-aa6d-20081fda869e')

In [0]:
%sql
select order_id, quantity, product_name, weight_kg from workspace.amazon_fullfilment_silver.order_items_silver 
join workspace.amazon_fullfilment_silver.products_silver on workspace.amazon_fullfilment_silver.order_items_silver.product_id = workspace.amazon_fullfilment_silver.products_silver.product_id and workspace.amazon_fullfilment_silver.products_silver.is_current = true
where order_id in ('e8905e2f-aab5-4bbd-9b5b-b9a4089330b1', '43eb70ec-ff11-4cf8-aa6d-20081fda869e')

In [0]:
# Insert into Bronze layer bin_events table
 
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType, IntegerType

# Define exactly what the data should look like
bin_schema = StructType([
    StructField("bin_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("item_count", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("current_weight", DoubleType(), True),
    StructField("employee_id", StringType(), True),
    StructField("updated_timestamp", TimestampType(), True)
])

# 1. Define your paths
bronze_bin_path = f"{bronze_volume_path}bin_events"

ingest_to_bronze("bin_events", bin_schema, f"{source_volume_path}bin_events", bronze_bin_path)

In [0]:
%sql
select * from workspace.amazon_fullfilment_bronze.bin_events

In [0]:
# process new records into the Silver layer bin table
from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp, current_date

def process_scd2_bins(microBatchDF, batchId):
    # 1. Get a list of order_ids that already exist in Silver
    # This prevents duplicating brand new orders
    target_table = "workspace.amazon_fullfilment_silver.bins"
    
    # 2. Split the micro-batch logic
    # Part A: The 'Insert' half (Always NULL merge_key)
    inserts_df = microBatchDF.withColumn("merge_key", F.lit(None))
    
    # Part B: The 'Update' half (Only for IDs that already exist in Silver)
    # This prevents the 'double insert' on the first run
    existing_ids = microBatchDF.sparkSession.table(target_table) \
        .filter("is_active = true") \
        .select("bin_id")
    
    updates_df = microBatchDF.join(existing_ids, "bin_id", "inner") \
        .withColumn("merge_key", F.col("bin_id"))
    
    # 3. Combine and Merge
    combined_df = inserts_df.unionByName(updates_df, allowMissingColumns=True)
    combined_df.createOrReplaceTempView("source_bins_view")

    microBatchDF.sparkSession.sql(f"""
        MERGE INTO {target_table} AS target
        USING source_bins_view AS source
        ON target.bin_id = source.merge_key AND target.is_active = true
        
        -- If we find the ID and status changed, expire the old one
        WHEN MATCHED AND target.status <> source.status AND target.order_id <> source.order_id THEN
          UPDATE SET target.is_active = false, target.end_date = source.updated_timestamp, target.updated_timestamp = source.updated_timestamp
          
        -- If it's the 'NULL' half of the union, it will always be NOT MATCHED -> Insert
        WHEN NOT MATCHED THEN
          INSERT (bins_sk, bin_id, order_id, item_count, status, current_weight, is_active, start_date, end_date, updated_timestamp)
          VALUES (md5(concat(source.bin_id, cast(source.updated_timestamp as STRING))), 
                  source.bin_id, source.order_id, source.item_count, source.status, source.current_weight, 
                  true, source.updated_timestamp, NULL, source.updated_timestamp)
    """)

    # 3. Start the Stream
query = (spark.readStream
    .table("workspace.amazon_fullfilment_bronze.bin_events")
    .filter("status IS NOT NULL") # Basic data quality
    .writeStream
    .foreachBatch(process_scd2_bins)
    .option("checkpointLocation", f"{silver_volume_path}/checkpoints/silver_bin_scd2")
    .trigger(availableNow=True) # Or .trigger(processingTime='1 minute') for 24/7
    .start())

In [0]:
%sql
select * from workspace.amazon_fullfilment_silver.bins

In [0]:

# update the bin_event table by emptying the bins based on available employee pull
# update the inventory and shelves when the orders from bins are completed and shipped
# undate the orders when they have been completed (shipped)

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col

# 1. Identify bins in 'Shipment' status for the current batch
shipping_bins = (spark.table("workspace.amazon_fullfilment_silver.bins")
                 .filter("is_active = true")
                 .filter(F.col("status") == 'Picking'))

print(f"Total active Picking Bins: {shipping_bins.count()}")

# 2. Identify available Packers (1 bin per Packer constraint)
packers_df = (spark.table("workspace.amazon_fullfilment_silver.employee")
              .filter(f"is_active = true AND job_role = 'Packer' AND shift_id = '{current_shift}'")
              .select("employee_id")
              .withColumn("packer_rank", F.row_number().over(Window.orderBy("employee_id"))))

print(f"Total active Pakers: {packers_df.count()}") 

# 3. Assign Packers and Calculate Random Processing Time
# Formula for random minutes: 2 + (rand() * 8)
ranked_bins = shipping_bins.withColumn("bin_rank", F.row_number().over(Window.orderBy("bin_id")))

shipped_assignments = (ranked_bins.join(packers_df, ranked_bins.bin_rank == packers_df.packer_rank, "inner")
                       .withColumn("random_minutes", (F.lit(2) + (F.rand(seed=42) * 8)).cast("int"))
                       .withColumn("shipped_time", F.expr("updated_timestamp + make_interval(0, 0, 0, 0, 0, random_minutes, 0)")))

print(f"Total active shipped_assignments: {shipped_assignments.count()}") 

# --- OUTPUT A: Bin Records ---
bin_shipped_records = (shipped_assignments.select(
    "bin_id",
    "order_id",
    F.lit(0).alias("item_count"),
    F.lit("Shipped").alias("status"),
    "current_weight",
    packers_df["employee_id"], 
    F.col("shipped_time").alias("updated_timestamp"),
  #  F.lit(True).alias("is_active"),
  #  F.lit(current_run_uuid).alias("_batch_id"),
    F.current_timestamp().alias("_ingest_timestamp")
))

display(bin_shipped_records)

# --- OUTPUT B: Order Records ---
# 1. First, create the exploded list of unique order IDs from the assignments
exploded_shipped_orders = (shipped_assignments
    .withColumn("order_id", F.explode(F.split(F.col("order_id"), ", ")))
    .select("order_id", "shipped_time")
    .dropDuplicates(["order_id"]))

# 2. Join back to the original Bronze orders table to get the missing fields
order_shipped_records = (exploded_shipped_orders
    .join(
        spark.table("workspace.amazon_fullfilment_silver.orders_silver").select("order_id", "customer_id", "orderdate", "status", "is_current"),
        #spark.table("workspace.amazon_fullfilment_silver.orders_silver").select("order_id", "customer_id", "orderdate", "status","_batch_id"),  
        "order_id", 
        "inner"
    ).filter(F.col("status") == "Picking").filter("is_current == true")
    #).filter(F.col("status") == "Picking").filter(F.col("_batch_id") == current_run_uuid)
    .select(
        "order_id",
        "customer_id",
        "orderdate",
        F.lit("Shipped").alias("status"),
        F.col("shipped_time").alias("updated_timestamp")
       # "is_current",
     #   F.lit(current_run_uuid).alias("_batch_id"),
     #   F.current_timestamp().alias("_ingest_timestamp")
    ))

display(order_shipped_records)

### repeat the above for the inventory table.  Then you need to use the shelf_id associated to set the status of the shelf to 'empty'. Then independently you can reduce the robot battery to %2. Check if the battery is less than %10 and send to to charging.



# 1. Identify the Orders that were just Shipped
# We use the order_shipped_records from the previous step
shipped_order_ids = order_shipped_records.select("order_id")

# 2. Identify the Inventory items associated with those orders
# We need to know which shelf_id they came from before we remove them
current_inventory = spark.table("workspace.amazon_fullfilment_silver.inventory_silver")
#current_inventory = spark.table(f"{catalog_name}.{database_name}.inventory").filter(F.col("_batch_id") == current_run_uuid)

inventory_to_remove = current_inventory.join(shipped_order_ids, "order_id", "inner")

# --- OUTPUT A: Inventory "Pop" Records ---
# Since this is Bronze (Append-only), we create records with 0 or negative quantity 
# to represent the removal, or simply treat the absence of these IDs in Silver as the "Pop".
# For this simulation, we will generate the "Post-Removal" state.
new_inventory_state = current_inventory.join(shipped_order_ids, "order_id", "left_anti")

# 3. Determine Shelf Status (Idle vs Picking)
# A shelf is 'idle' ONLY if it no longer appears in the new_inventory_state
active_shelves = new_inventory_state.select(F.col("shelf_id").alias("active_shelf_id")).distinct()

# Get all shelves that were previously associated with our shipped orders
shelves_to_check = inventory_to_remove.select("shelf_id").distinct()

shelf_updates = (shelves_to_check.join(active_shelves, shelves_to_check.shelf_id == active_shelves.active_shelf_id, "left_outer")
    .withColumn("status", F.when(F.col("active_shelf_id").isNull(), "idle").otherwise("Picking"))
    # Join back to the original shelf table to get robot_id and capacity
    .join(spark.table("workspace.amazon_fullfilment_silver.shelves").select("shelf_id", "robot_id", "max_weight_capacity"), "shelf_id")
    .select(
        "shelf_id",
        "robot_id",
        "max_weight_capacity",
        "status",
        F.current_timestamp().alias("updated_timestamp")
      #  F.lit(True).alias("is_active"),
       # F.lit(current_run_uuid).alias("_batch_id"),
       # F.current_timestamp().alias("_ingest_timestamp")
    ))

# 4. Filter Inventory to only show what is LEAVING (The "Pop" action)
# In many Bronze architectures, you record the "Deletion" event
inventory_removal_records = (inventory_to_remove
    .withColumn("quantity", F.lit(0)) # Setting quantity to 0 indicates it's gone
    .select(
        "shelf_id",
        "product_id",
        "quantity",
        "order_id"
      #  F.lit(True).alias("is_active"),
        #F.lit(current_run_uuid).alias("_batch_id"),
       # F.current_timestamp().alias("_ingest_timestamp")
    ))

display(shelf_updates)
display(inventory_removal_records)
shelf_distinct_updates_df = shelf_updates.distinct()


# Save to source folder
(bin_shipped_records.write
 .format("csv")
 .option("header",True)
 .option("delimiter",",")
 .mode("append")
 .save(f"{source_volume_path}/bin_events"))
(inventory_removal_records
 .write.format("csv")
 .option("header",True)
 .option("delimiter",",")
 .mode("append")
 .save(f"{source_volume_path}/inventory"))
(shelf_distinct_updates_df
 .write.format("csv")
 .option("header",True)
 .option("delimiter",",")
 .mode("append")
 .save(f"{source_volume_path}/shelves_events"))
(order_shipped_records
 .drop("_metadata")
 .write.format("csv")
 .option("header",True)
 .option("delimiter",",")
 .mode("append")
 .save(f"{source_volume_path}/orders"))


# --- OUTPUT B: Order Records ---
# order_shipped_records = (shipped_assignments
#     .withColumn("single_order_id", F.explode(F.split(F.col("order_id"), ", ")))
#     .select(
#         F.col("single_order_id").alias("order_id"),
#         #"customer_id",
#         #order_date
#         F.lit("Shipped").alias("status"),
#         F.col("shipped_time").alias("updated_timestamp"),
#         F.lit(current_run_uuid).alias("_batch_id"),
#         F.current_timestamp().alias("_ingest_timestamp")
#     ).dropDuplicates(["order_id"]))

#display(bin_shipped_records.select("bin_id", "order_id","employee_id", "updated_timestamp", "status"))

#display(order_shipped_records)

In [0]:
%sql
select * from workspace.amazon_fullfilment_silver.employee where is_active = true AND job_role = 'Packer'


In [0]:
# Insert into Bronze layer bin_events table
 
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType, IntegerType

# Define exactly what the data should look like
bin_schema = StructType([
    StructField("bin_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("item_count", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("current_weight", DoubleType(), True),
    StructField("employee_id", StringType(), True),
    StructField("updated_timestamp", TimestampType(), True)
])

# 1. Define your paths
bronze_bin_path = f"{bronze_volume_path}bin_events"

ingest_to_bronze("bin_events", bin_schema, f"{source_volume_path}bin_events", bronze_bin_path)

In [0]:
# process new records into the Silver layer bin table
from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp, current_date

def process_scd2_bins(microBatchDF, batchId):
    # 1. Get a list of order_ids that already exist in Silver
    # This prevents duplicating brand new orders
    target_table = "workspace.amazon_fullfilment_silver.bins"
    
    # 2. Split the micro-batch logic
    # Part A: The 'Insert' half (Always NULL merge_key)
    inserts_df = microBatchDF.withColumn("merge_key", F.lit(None))
    
    # Part B: The 'Update' half (Only for IDs that already exist in Silver)
    # This prevents the 'double insert' on the first run
    existing_ids = microBatchDF.sparkSession.table(target_table) \
        .filter("is_active = true") \
        .select("bin_id")
    
    updates_df = microBatchDF.join(existing_ids, "bin_id", "inner") \
        .withColumn("merge_key", F.col("bin_id"))
    
    # 3. Combine and Merge
    combined_df = inserts_df.unionByName(updates_df, allowMissingColumns=True)
    combined_df.createOrReplaceTempView("source_bins_view")

    microBatchDF.sparkSession.sql(f"""
        MERGE INTO {target_table} AS target
        USING source_bins_view AS source
        ON target.bin_id = source.merge_key AND target.is_active = true
        
        -- If we find the ID and status changed, expire the old one
        WHEN MATCHED AND target.status <> source.status AND target.order_id <> source.order_id THEN
          UPDATE SET target.is_active = false, target.end_date = source.updated_timestamp, target.updated_timestamp = source.updated_timestamp
          
        -- If it's the 'NULL' half of the union, it will always be NOT MATCHED -> Insert
        WHEN NOT MATCHED THEN
          INSERT (bins_sk, bin_id, order_id, item_count, status, current_weight, is_active, start_date, end_date, updated_timestamp)
          VALUES (md5(concat(source.bin_id, cast(source.updated_timestamp as STRING))), 
                  source.bin_id, source.order_id, source.item_count, source.status, source.current_weight, 
                  true, source.updated_timestamp, NULL, source.updated_timestamp)
    """)

    # 3. Start the Stream
query = (spark.readStream
    .table("workspace.amazon_fullfilment_bronze.bin_events")
    .filter("status IS NOT NULL") # Basic data quality
    .writeStream
    .foreachBatch(process_scd2_bins)
    .option("checkpointLocation", f"{silver_volume_path}/checkpoints/silver_bin_scd2")
    .trigger(availableNow=True) # Or .trigger(processingTime='1 minute') for 24/7
    .start())

In [0]:
# Insert into Bronze layer Inventory table
 
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType, IntegerType

# Define exactly what the data should look like
inventory_schema = StructType([
    StructField("shelf_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("order_id", StringType(), True)
])

# 1. Define your paths
bronze_inventory_path = f"{bronze_volume_path}inventory"

ingest_to_bronze("inventory", inventory_schema, f"{source_volume_path}inventory", bronze_inventory_path)

In [0]:
# process new records into the Silver layer inventory table
from pyspark.sql import functions as F

def process_scd1_inventory(microBatchDF, batchId):
    target_table = "workspace.amazon_fullfilment_silver.inventory_silver"
    
    # 1. Deduplicate based on the COMPOSITE natural key
    # This ensures we only have one row per shelf/product combo per batch
    source_df = microBatchDF.dropDuplicates(["shelf_id", "product_id"])
    source_df.createOrReplaceTempView("source_inventory_view")

    # 2. Updated Merge Logic
    microBatchDF.sparkSession.sql(f"""
        MERGE INTO {target_table} AS target
        USING source_inventory_view AS source
        ON target.shelf_id = source.shelf_id 
           AND target.product_id = source.product_id
        
        -- We only update the 'Quantity' or 'Order_ID' if the composite key matches
        WHEN MATCHED AND (
            source.quantity <> target.quantity OR 
            source.order_id <> target.order_id
        ) THEN
            UPDATE SET 
                target.quantity = source.quantity,
                target.order_id = source.order_id,
                target.updated_timestamp = current_timestamp()
          
        WHEN NOT MATCHED THEN
          INSERT (
            inventory_sk, 
            shelf_id, 
            product_id, 
            quantity, 
            order_id, 
            updated_timestamp
          )
          VALUES (
            -- IMPORTANT: The SK must now be a hash of BOTH columns
            md5(concat(source.shelf_id, source.product_id)), 
            source.shelf_id, 
            source.product_id, 
            source.quantity, 
            source.order_id, 
            current_timestamp()
          )
    """)
    # 3. Start the Stream
query = (spark.readStream
    .table("workspace.amazon_fullfilment_bronze.inventory")
    .filter("shelf_id IS NOT NULL") # Basic data quality
    .writeStream
    .foreachBatch(process_scd1_inventory)
    .option("checkpointLocation", f"{silver_volume_path}/checkpoints/silver_inventory_scd1")
    .trigger(availableNow=True) # Or .trigger(processingTime='1 minute') for 24/7
    .start())

In [0]:
%sql
select * from workspace.amazon_fullfilment_silver.inventory_silver

In [0]:
# Insert into Bronze layer Shelves table
 
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType, IntegerType

# Define exactly what the data should look like
shelve_schema = StructType([
    StructField("shelf_id", StringType(), True),
    StructField("robot_id", StringType(), True),
    StructField("max_weight_capacity", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("updated_timestamp", TimestampType(), True)
])

# 1. Define your paths
bronze_shelves_path = f"{bronze_volume_path}shelves_events"

ingest_to_bronze("shelves_events", shelve_schema, f"{source_volume_path}shelves_events", bronze_shelves_path)

In [0]:
# process new records into the Silver layer shelves table
from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp, current_date

def process_scd2_shelves(microBatchDF, batchId):
    # 1. Get a list of order_ids that already exist in Silver
    # This prevents duplicating brand new orders
    target_table = "workspace.amazon_fullfilment_silver.shelves"
    
    # 2. Split the micro-batch logic
    # Part A: The 'Insert' half (Always NULL merge_key)
    inserts_df = microBatchDF.withColumn("merge_key", F.lit(None))
    
    # Part B: The 'Update' half (Only for IDs that already exist in Silver)
    # This prevents the 'double insert' on the first run
    existing_ids = microBatchDF.sparkSession.table(target_table) \
        .filter("is_active = true") \
        .select("shelf_id")
    
    updates_df = microBatchDF.join(existing_ids, "shelf_id", "inner") \
        .withColumn("merge_key", F.col("shelf_id"))
    
    # 3. Combine and Merge
    combined_df = inserts_df.unionByName(updates_df, allowMissingColumns=True)
    combined_df.createOrReplaceTempView("source_shelves_view")

    microBatchDF.sparkSession.sql(f"""
        MERGE INTO {target_table} AS target
        USING source_shelves_view AS source
        ON target.shelf_id = source.merge_key AND target.is_active = true
        
        -- If we find the ID and status changed, expire the old one
        WHEN MATCHED AND target.status <> source.status THEN
          UPDATE SET target.is_active = false, target.end_date = source.updated_timestamp, target.updated_timestamp = source.updated_timestamp
          
        -- If it's the 'NULL' half of the union, it will always be NOT MATCHED -> Insert
        WHEN NOT MATCHED THEN
          INSERT (shelf_sk, shelf_id, robot_id, max_weight_capacity, status, is_active, start_date, end_date, updated_timestamp)
          VALUES (md5(concat(source.shelf_id, cast(source.updated_timestamp as STRING))), 
                  source.shelf_id, source.robot_id, source.max_weight_capacity, source.status,  
                  true, source.updated_timestamp, NULL, source.updated_timestamp)
    """)

    # 3. Start the Stream
query = (spark.readStream
    .table("workspace.amazon_fullfilment_bronze.shelves_events")
    .filter("status IS NOT NULL") # Basic data quality
    .writeStream
    .foreachBatch(process_scd2_shelves)
    .option("checkpointLocation", f"{silver_volume_path}/checkpoints/silver_shelves_scd2")
    .trigger(availableNow=True) # Or .trigger(processingTime='1 minute') for 24/7
    .start())

In [0]:
# Insert into Bronze layer order table
 
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType, IntegerType

# Define exactly what the data should look like
orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("orderdate", DateType(), True),
    StructField("status", StringType(), True),
    StructField("updated_timestamp", TimestampType(), True)
])

# 1. Define your paths
bronze_orders_path = f"{bronze_volume_path}orders"

ingest_to_bronze("orders", orders_schema, f"{source_volume_path}orders", bronze_orders_path)

In [0]:
# process new records into the Silver layer orders table
from pyspark.sql import functions as F

def process_scd2_orders(microBatchDF, batchId):
    # 1. Get a list of order_ids that already exist in Silver
    # This prevents duplicating brand new orders
    target_table = "workspace.amazon_fullfilment_silver.orders_silver"
    
    # 2. Split the micro-batch logic
    # Part A: The 'Insert' half (Always NULL merge_key)
    inserts_df = microBatchDF.withColumn("merge_key", F.lit(None))
    
    # Part B: The 'Update' half (Only for IDs that already exist in Silver)
    # This prevents the 'double insert' on the first run
    existing_ids = microBatchDF.sparkSession.table(target_table) \
        .filter("is_current = true") \
        .select("order_id")
    
    updates_df = microBatchDF.join(existing_ids, "order_id", "inner") \
        .withColumn("merge_key", F.col("order_id"))
    
    # 3. Combine and Merge
    combined_df = inserts_df.unionByName(updates_df, allowMissingColumns=True)
    combined_df.createOrReplaceTempView("source_view")

    microBatchDF.sparkSession.sql(f"""
        MERGE INTO {target_table} AS target
        USING source_view AS source
        ON target.order_id = source.merge_key AND target.is_current = true
        
        -- If we find the ID and status changed, expire the old one
        WHEN MATCHED AND target.status <> source.status THEN
          UPDATE SET target.is_current = false, target.end_date = source.updated_timestamp
          
        -- If it's the 'NULL' half of the union, it will always be NOT MATCHED -> Insert
        WHEN NOT MATCHED THEN
          INSERT (order_sk, order_id, customer_id, orderdate, status, updated_timestamp, is_current, start_date, end_date)
          VALUES (md5(concat(source.order_id, cast(source.updated_timestamp as STRING))), 
                  source.order_id, source.customer_id, source.orderdate, source.status, 
                  source.updated_timestamp, true, source.updated_timestamp, NULL)
    """)

    # 3. Start the Stream
query = (spark.readStream
    .table("workspace.amazon_fullfilment_bronze.orders")
    .filter("status IS NOT NULL") # Basic data quality
    .writeStream
    .foreachBatch(process_scd2_orders)
    .option("checkpointLocation", f"{bronze_volume_path}/checkpoints/silver_orders_scd2")
    .trigger(availableNow=True) # Or .trigger(processingTime='1 minute') for 24/7
    .start())

In [0]:
%sql
select * from workspace.amazon_fullfilment_silver.orders_silver

In [0]:
# update the robot status if their shelves are now empty

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Determine Robot Status based on Shelf occupancy
# (Using the same robot_status_check logic from the previous step)

#robot_status_check = (all_shelf_statuses.groupBy("robot_id")
robot_status_check = (shelf_updates.groupBy("robot_id")
    .agg(F.collect_set("status").alias("status_set"))
    .withColumn("calculated_status", 
        F.when((F.size(F.col("status_set")) == 1) & (F.array_contains(F.col("status_set"), "idle")), "Idle")
        .otherwise("Picking")))

# 2. Apply Battery Drain, Low-Battery Logic, and Final Status
robot_updates_df = (robot_status_check
    .join(spark.table(f"{catalog_name}.{database_name}.robot_events"), "robot_id")
    # Get the most recent state for each robot
    .withColumn("row_num", F.row_number().over(Window.partitionBy("robot_id").orderBy(F.col("event_timestamp").desc())))
    .filter("row_num = 1")
    .withColumn("new_battery_level", F.greatest(F.lit(0), F.col("battery_level") - 2))
    .withColumn("final_status", 
        F.when(F.col("new_battery_level") < 10, "Charging")
        .otherwise(F.col("calculated_status")))
    .select(
        "robot_id",
        F.col("final_status").alias("status"),
        "max_weight_capacity",
        F.col("new_battery_level").alias("battery_level"),
        F.current_timestamp().alias("event_timestamp"),
        F.lit(current_run_uuid).alias("_batch_id"),
        F.current_timestamp().alias("_ingest_timestamp")
    ))

# 3. Filter for available (Idle) or maintaining (Charging) robots
final_robot_output_df = robot_updates_df.filter(F.col("status").isin("Idle", "Charging"))

display(final_robot_output_df)

(final_robot_output_df
 .drop("_metadata")
 .write.format("csv")
 .option("header",True)
 .option("delimiter",",")
 .mode("append")
 .save(f"{source_volume_path}/robot_events"))

In [0]:
# Insert into Bronze layer robot_events table
 
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType, IntegerType

# Define exactly what the data should look like
robot_events_schema = StructType([
    StructField("robot_id", StringType(), True),
    StructField("status", StringType(), True),
    StructField("max_weight_capacity", DoubleType(), True),
    StructField("battery_level", IntegerType(), True),
    StructField("event_timestamp", TimestampType(), True)
])

# 1. Define your paths
bronze_robot_events_path = f"{bronze_volume_path}robot_events"

ingest_to_bronze("robot_events", robot_events_schema, f"{source_volume_path}robot_events", bronze_robot_events_path)

In [0]:
# process new records into the Silver layer robots table
from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp, current_date

def process_scd2_robots(microBatchDF, batchId):
    # 1. Get a list of order_ids that already exist in Silver
    # This prevents duplicating brand new orders
    target_table = "workspace.amazon_fullfilment_silver.robots"
    
    # 2. Split the micro-batch logic
    # Part A: The 'Insert' half (Always NULL merge_key)
    inserts_df = microBatchDF.withColumn("merge_key", F.lit(None))
    
    # Part B: The 'Update' half (Only for IDs that already exist in Silver)
    # This prevents the 'double insert' on the first run
    existing_ids = microBatchDF.sparkSession.table(target_table) \
        .filter("is_active = true") \
        .select("robot_id")
    
    updates_df = microBatchDF.join(existing_ids, "robot_id", "inner") \
        .withColumn("merge_key", F.col("robot_id"))
    
    # 3. Combine and Merge
    combined_df = inserts_df.unionByName(updates_df, allowMissingColumns=True)
    combined_df.createOrReplaceTempView("source_robot_view")

    microBatchDF.sparkSession.sql(f"""
        MERGE INTO {target_table} AS target
        USING source_robot_view AS source
        ON target.robot_id = source.merge_key AND target.is_active = true
        
        -- If we find the ID and status changed, expire the old one
        WHEN MATCHED AND target.status <> source.status THEN
          UPDATE SET target.is_active = false, target.end_date = source.event_timestamp, target.updated_timestamp = source.event_timestamp
          
        -- If it's the 'NULL' half of the union, it will always be NOT MATCHED -> Insert
        WHEN NOT MATCHED THEN
          INSERT (robot_sk, robot_id, robot_type, max_weight_capacity, max_speed_mps, status, current_battery_pct, last_maintenance_date,  is_active, start_date, end_date, updated_timestamp)
          VALUES (md5(concat(source.robot_id, cast(source.event_timestamp as STRING))), 
                  source.robot_id, 'Hercules', source.max_weight_capacity, 10.0, source.status, source.battery_level, current_date(), 
                  true, source.event_timestamp, NULL, source.event_timestamp)
    """)

    # 3. Start the Stream
query = (spark.readStream
    .table("workspace.amazon_fullfilment_bronze.robot_events")
    .filter("status IS NOT NULL") # Basic data quality
    .writeStream
    .foreachBatch(process_scd2_robots)
    .option("checkpointLocation", f"{silver_volume_path}/checkpoints/silver_robots_scd2")
    .trigger(availableNow=True) # Or .trigger(processingTime='1 minute') for 24/7
    .start())

In [0]:
%sql
select * from workspace.amazon_fullfilment_silver.orders_silver

In [0]:
%sql
select * from workspace.amazon_fullfilment_bronze.orders

In [0]:
%sql
describe history workspace.amazon_fullfilment_bronze.orders

In [0]:
%sql
select * from workspace.amazon_fullfilment_silver.order_items_silver

In [0]:
%sql
describe history workspace.amazon_fullfilment_silver.order_items_silver

In [0]:
%sql
describe history workspace.amazon_fullfilment_bronze.order_items

In [0]:
display(f"{source_volume_path}/order_items")

In [0]:
%sql
select * from csv.`/Volumes/workspace/default/amazon_fullfilment/source/order_items`